# Notebook 03 — Theoretical curve r(D) vs observed scatter

Goal: reproduce the analytical curve fit from the Quiescence
Signature derivation:
  r(D) ≈ 1 / sqrt(1 + tau D)

Inputs:
  - Cached arrays from `dcps/cache/eke_eddy_resolving/`
    (rl_mean and EKE on the matched 1/2-deg NA grid)

Outputs:
  - Fitted tau
  - Residual MSE

Acceptance: tau in [100, 200], MSE in [0.02, 0.04].

In [ ]:
import sys
from pathlib import Path

REPO = Path('/home/bijanf/Documents/NEW_Theory')
sys.path.insert(0, str(REPO / 'quiescence_toolkit' / 'src'))

from theoretical_curve import fit_and_score
import xarray as xr
import numpy as np

rl = xr.open_dataset(REPO / 'dcps/cache/eke_eddy_resolving/rl_mean_2thdeg.nc')
eke = xr.open_dataset(REPO / 'dcps/cache/eke_eddy_resolving/eke_2thdeg.nc')

r_arr = rl[list(rl.data_vars)[0]].values.ravel()
D_arr = eke[list(eke.data_vars)[0]].values.ravel()
valid = np.isfinite(r_arr) & np.isfinite(D_arr) & (D_arr > 0)

tau, r_fit, mse = fit_and_score(D_arr[valid], r_arr[valid], 'simple')
print(f'fitted tau = {tau:.1f}')
print(f'residual MSE = {mse:.4f}')

assert 100 <= tau <= 200, f'tau should be in [100, 200], got {tau}'
assert 0.02 <= mse <= 0.04, f'MSE should be in [0.02, 0.04], got {mse}'
print('PASS')